# CrewAI Implementation: Agentic Workflows with Structured Outputs

This notebook demonstrates how to build agentic workflows using **CrewAI** with native Pydantic structured outputs (`output_pydantic`), replacing standard LangChain workflows.

### 1. Package Installations & Imports

In [1]:
# !pip install crewai crewai-tools langchain-community yfinance pandas python-dotenv

import os
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# CrewAI imports
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from langchain_community.document_loaders import WeatherDataLoader

# Load environment variables from .env file
load_dotenv()

C:\Users\hp\AppData\Local\Temp\ipykernel_2328\771382003.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WeatherDataLoader


True

### 2. Environment Configuration & LLM Initialization
CrewAI uses LiteLLM under the hood. We define our models using prefix identifiers (`groq/` and `huggingface/`).

In [ ]:
# Set Environment Variables
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENWEATHERMAP_API_KEY"] = os.getenv("OPENWEATHERMAP_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Initialize Groq LLMs
qwen_llm = LLM(
    model="groq/qwen/qwen3-32b",
    api_key=os.environ["GROQ_API_KEY"]
)

llama_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

# Initialize HuggingFace LLM
hf_llm = LLM(
    model="huggingface/deepseek-ai/DeepSeek-V4-Flash",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    temperature=0.1,
    max_tokens=512
)

### 3. Weather Tools & Pydantic Schemas

In [ ]:
# --- TOOLS ---
@tool("get_current_weather")
def get_current_weather(location: str) -> str:
    """Get the current weather of a given location.
    
    Args:
        location: place of which weather data you want
    """
    loader = WeatherDataLoader.from_params(
        [location], openweathermap_api_key=os.getenv("OPENWEATHERMAP_API_KEY")
    )
    documents = loader.load()
    return documents[0].page_content

# --- PYDANTIC SCHEMAS ---
class Weather(BaseModel):
    place: str = Field(description="Specified place for weather details")
    temperature: float = Field(description="Temperature of the specified location")
    humidity: float = Field(description="Humidity of the specified location")
    wind: float = Field(description="Wind Speed of the specified location")

class MultiWeather(BaseModel):
    weather: list[Weather] = Field(description="List of weather details of specified locations")

### 4. Single Location Weather Agent

In [ ]:
weather_agent = Agent(
    role="Weather Reporter",
    goal="Accurately fetch and structure weather data for any requested location.",
    backstory="You are a meteorological assistant specialized in extracting precise weather conditions.",
    tools=[get_current_weather],
    llm=qwen_llm,
    verbose=True
)

weather_task = Task(
    description="Tell me the weather of Pune.",
    expected_output="The temperature, humidity, and wind speed details of Pune.",
    agent=weather_agent,
    output_pydantic=Weather
)

crew = Crew(
    agents=[weather_agent],
    tasks=[weather_task],
    process=Process.sequential
)

result = crew.kickoff()

# Extract Pydantic model and display as DataFrame
structured_response = result.pydantic
df_single = pd.DataFrame([structured_response.model_dump()])
df_single

### 5. Multiple Locations Weather Agent

In [ ]:
places = "Hyderabad, Bengaluru, Chennai, Kolkata, Mumbai"

multi_weather_task = Task(
    description=f"Tell me weather of {places}",
    expected_output="A structured list containing exact weather details for all requested cities.",
    agent=weather_agent,
    output_pydantic=MultiWeather
)

multi_crew = Crew(
    agents=[weather_agent],
    tasks=[multi_weather_task],
    process=Process.sequential
)

multi_result = multi_crew.kickoff()

# Convert structured output list to DataFrame
df_multi = pd.DataFrame([w.model_dump() for w in multi_result.pydantic.weather])
df_multi

### 6. Stock Market Tools & Schemas

In [ ]:
# --- STOCK TOOLS ---
@tool("get_ticker_from_name")
def get_ticker_from_name(company_name: str) -> str:
    """Specify the company name of which stock code / scrip code you want.
    Returns the stock symbol corresponding to the company name given.
    """
    search = yf.Search(company_name, max_results=5)
    if search.quotes:
        for result in search.quotes:
            answer = f"Symbol: {result['symbol']} | Name: {result['shortname']} | Exchange: {result['exchDisp']}"
            return answer
        return search.quotes[0]['symbol']
    return "No ticker found."

@tool("get_stock_price")
def get_stock_price(ticker: str) -> float:
    """Specify the stock code / scrip code of which you want the price.
    Returns the current price of the stock.
    """
    tck = yf.Ticker(ticker)
    info = tck.info
    return info.get('currentPrice')

# --- STOCK SCHEMA ---
class Stock(BaseModel):
    company: str = Field(description="Company specified for stock price")
    price: float = Field(description="Latest stock price on National Stock Exchange")

### 7. Stock Market Execution (Groq Llama-3.3)

In [ ]:
stock_agent_groq = Agent(
    role="Stock Market Observer",
    goal="Identify correct ticker symbols and fetch exact current stock prices.",
    backstory="You are a financial data analyst skilled at reading stock market feeds.",
    tools=[get_ticker_from_name, get_stock_price],
    llm=llama_llm,
    verbose=True
)

stock_task_groq = Task(
    description="Tell me latest stock price of Indian Oil on NSE.",
    expected_output="The company name and exact latest stock price.",
    agent=stock_agent_groq,
    output_pydantic=Stock
)

stock_crew_groq = Crew(
    agents=[stock_agent_groq],
    tasks=[stock_task_groq],
    process=Process.sequential
)

stock_result_groq = stock_crew_groq.kickoff()
df_stock_groq = pd.DataFrame([stock_result_groq.pydantic.model_dump()])
df_stock_groq

### 8. Stock Market Execution (HuggingFace Endpoint)

In [ ]:
stock_agent_hf = Agent(
    role="Stock Market Observer",
    goal="Identify correct ticker symbols and fetch exact current stock prices.",
    backstory="You are a financial data analyst skilled at reading stock market feeds.",
    tools=[get_ticker_from_name, get_stock_price],
    llm=hf_llm,
    verbose=True
)

stock_task_hf = Task(
    description="Tell me stock price of Indian Oil on NSE.",
    expected_output="The company name and exact latest stock price.",
    agent=stock_agent_hf,
    output_pydantic=Stock
)

stock_crew_hf = Crew(
    agents=[stock_agent_hf],
    tasks=[stock_task_hf],
    process=Process.sequential
)

stock_result_hf = stock_crew_hf.kickoff()
df_stock_hf = pd.DataFrame([stock_result_hf.pydantic.model_dump()])
df_stock_hf